# 03 – Modell Tanítás és 5-Fold Keresztvalidáció (Márka Osztályozás)

**Projekt:** Nike / Adidas / Converse cipőmárkák képosztályozása  
**Cél:** Mélytanulási modell (Transfer Learning / ResNet / MobileNetV2) építése és betanítása az előkészített adathalmazon **Stratified 5-Fold Keresztvalidációval**, a modell jósági mutatóinak (Accuracy, Macro F1) mérésével és elmentésével.

**Bemenet:** `../results/brand/dataset_with_folds.csv` és a képfájlok  
**Kimenet:** Foldonkénti validációs eredmények, átlagos teljesítmény, mentett modell súlyok.

## 1. Könyvtárak Betöltése és Beállítások

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Globális beállítások
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

IMG_SIZE = (224, 224)
BATCH_SIZE = 8
#16
EPOCHS = 10
#20
LEARNING_RATE = 1e-4
K_FOLDS = 5

PROJECT_ROOT = Path("..").resolve()
MANIFEST_PATH = PROJECT_ROOT / "results" / "brand" / "dataset_with_folds.csv"
MODEL_SAVE_DIR = PROJECT_ROOT / "results" / "models_brand"
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Preprocessing Manifest betöltése
manifest = pd.read_csv(MANIFEST_PATH)
print(f"Manifest betöltve: {len(manifest)} kép.")
print(f"Osztályok: {manifest['label'].unique()}")

## 2. Segédfüggvények: DataGenerator és Modell Építés

In [ ]:
def build_model(num_classes=3):
    """Előre betanított MobileNetV2 alapú Transfer Learning modell építése."""
    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
    )
    base_model.trainable = False  # Alaprétegek fagyasztása kezdetben

    inputs = layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    model.compile(
        optimizer=optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Data Augmentation a tanító halmazhoz
train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Nincs augmentation a validációs halmazhoz
val_datagen = ImageDataGenerator()

## 3. Stratified 5-Fold Keresztvalidációs Ciklus

In [ ]:
# Csak a tanító-validációs adatsor használata (is_test == False)
cv_data = manifest[manifest["is_test"] == False].copy()

fold_metrics = []
all_val_predictions = []
all_val_targets = []

for fold in range(K_FOLDS):
    print(f"\n=========================================")
    print(f"          FOLD {fold + 1} / {K_FOLDS} TANÍTÁSA          ")
    print(f"=========================================")
    
    train_df = cv_data[cv_data["fold"] != fold].copy()
    val_df = cv_data[cv_data["fold"] == fold].copy()
    
    # Data Flow a Dataframe-ből
    train_generator = train_datagen.flow_from_dataframe(
        dataframe=train_df,
        x_col="path",
        y_col="label",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        shuffle=True,
        seed=RANDOM_STATE
    )
    
    val_generator = val_datagen.flow_from_dataframe(
        dataframe=val_df,
        x_col="path",
        y_col="label",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        shuffle=False
    )
    
    model = build_model(num_classes=len(train_generator.class_indices))
    
    model_checkpoint_path = MODEL_SAVE_DIR / f"best_model_fold_{fold}.h5"
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        ModelCheckpoint(filepath=str(model_checkpoint_path), monitor='val_loss', save_best_only=True, verbose=0)
    ]
    
    history = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1
    )
    
    # Validáció kiértékelése
    val_generator.reset()
    val_preds_prob = model.predict(val_generator)
    val_preds = np.argmax(val_preds_prob, axis=1)
    val_true = val_generator.classes
    
    fold_acc = accuracy_score(val_true, val_preds)
    fold_f1 = f1_score(val_true, val_preds, average='macro')
    
    fold_metrics.append({
        'fold': fold + 1,
        'accuracy': fold_acc,
        'macro_f1': fold_f1
    })
    
    print(f"Fold {fold + 1} Eredmény -> Accuracy: {fold_acc:.4f} | Macro F1: {fold_f1:.4f}")
    
    all_val_predictions.extend(val_preds)
    all_val_targets.extend(val_true)

## 4. Keresztvalidáció Összegzése és Eredmények Dokumentálása

In [ ]:
results_df = pd.DataFrame(fold_metrics)
display(results_df)

mean_acc = results_df['accuracy'].mean()
std_acc = results_df['accuracy'].std()
mean_f1 = results_df['macro_f1'].mean()
std_f1 = results_df['macro_f1'].std()

print("\n=========================================")
print(f"  5-FOLD KERESZTVALIDÁCIÓS ÁTLAGEREDMÉNY  ")
print(f"=========================================")
print(f"Átlagos Accuracy: {mean_acc:.4f} (+/- {std_acc:.4f})")
print(f"Átlagos Macro F1: {mean_f1:.4f} (+/- {std_f1:.4f})")

# Eredmények elmentése JSON-be
summary_metrics = {
    'mean_accuracy': mean_acc,
    'std_accuracy': std_acc,
    'mean_macro_f1': mean_f1,
    'std_macro_f1': std_f1,
    'folds': fold_metrics
}

with open(PROJECT_ROOT / "results" / "cv_results_brand.json", "w", encoding="utf-8") as f:
    json.dump(summary_metrics, f, ensure_ascii=False, indent=4)

## 5. Tévesztési Mátrix (Confusion Matrix) Összesített Validációra

In [ ]:
class_labels = list(train_generator.class_indices.keys())
cm = confusion_matrix(all_val_targets, all_val_predictions)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_labels, yticklabels=class_labels)
plt.title("Összesített Keresztvalidációs Tévesztési Mátrix")
plt.xlabel("Predikált Osztály")
plt.ylabel("Valódi Osztály")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "results" / "confusion_matrix_cv_brand.png")
plt.show()

print("Classification Report a teljes validációra:")
print(classification_report(all_val_targets, all_val_predictions, target_names=class_labels))